In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

import dagshub
dagshub.init(repo_owner="AndriaMakharadze", repo_name="IEEE_Fraud_Detection_AM", mlflow=True)

Accessing as AndriaMakharadze

Initialized MLflow to track repo "AndriaMakharadze/IEEE_Fraud_Detection_AM"

Repository AndriaMakharadze/IEEE_Fraud_Detection_AM initialized!

In [2]:
train_transaction = pd.read_csv("../data/train_transaction.csv")
train_identity = pd.read_csv("../data/train_identity.csv")

df = train_transaction.merge(train_identity, on="TransactionID", how="left")
df = df.drop(columns=["TransactionID"], errors="ignore")

y = df["isFraud"]
X = df.drop(columns=["isFraud"])

X = X.sample(150000, random_state=42)
y = y.loc[X.index]

Cleaning

In [3]:
mlflow.set_experiment("XGBoost_Training")

with mlflow.start_run(run_name="XGBoost_Cleaning"):
    null_thresh = 0.8
    cols_to_drop = [c for c in X.columns if X[c].isnull().mean() > null_thresh]
    X = X.drop(columns=cols_to_drop)

    mlflow.log_param("null_threshhold", null_thresh)
    mlflow.log_param("cols_dropped", len(cols_to_drop))
    mlflow.log_metric("cols_remaining", X.shape[1])

    print(f"Dropped {len(cols_to_drop)} high-null columns. Remaining: {X.shape[1]}")

2026/05/04 14:01:26 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Training' does not exist. Creating a new experiment.


Dropped 74 high-null columns. Remaining: 358
🏃 View run XGBoost_Cleaning at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/1/runs/8f26f165dc004dce806caca15c357f02
🧪 View experiment at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/1


Feature Engineering

In [4]:
with mlflow.start_run(run_name="XGBoost_FeatureEngineering"):
    X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
    X["hour_of_day"] = (X["TransactionDT"] / 3600).astype(int) % 24
    X["null_count"] = X.isnull().sum(axis=1)

    new_features = ["TransactionAmt_log", "hour_of_day", "null_count"]
    mlflow.log_param("new_features", new_features)
    mlflow.log_metric("total_cols_after_eng", X.shape[1])
    print("Feature Engineering Done: ", new_features)

Feature Engineering Done:  ['TransactionAmt_log', 'hour_of_day', 'null_count']
🏃 View run XGBoost_FeatureEngineering at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/1/runs/2d7fb50ca18342be82d87f22bcbe98e1
🧪 View experiment at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/1


Feature Selection

In [5]:
with mlflow.start_run(run_name="XGBoost_FeatureSelection"):
    num_only = X.select_dtypes(include=["int64", "float64"]).fillna(0)
    corr_matrix = num_only.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
    X = X.drop(columns=drop_corr, errors="ignore")

    mlflow.log_param("corr_threshold", 0.95)
    mlflow.log_metric("cols_dropped_corr", len(drop_corr))
    mlflow.log_metric("cols_remaining", X.shape[1])
    print(f"Dropped {len(drop_corr)} correlated columns. Remaining: {X.shape[1]}")

Dropped 109 correlated columns. Remaining: 252
🏃 View run XGBoost_FeatureSelection at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/1/runs/a1d27f2c900e45908fdaa90f4d7be884
🧪 View experiment at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/1


In [6]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])

cat_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])

preprocessor = ColumnTransformer([("num", num_pipeline, num_cols), ("cat", cat_pipeline, cat_cols)])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Training

In [7]:
mlflow.set_experiment("XGBoost_Training")

with mlflow.start_run(run_name="XGBooxt_Training"):
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            eval_metric="auc",
            random_state=42,
            n_jobs=-1
        ))
    ])

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("auc", auc)

    mlflow.sklearn.log_model(pipeline, "pipeline_model", registered_model_name="XGBoost_FraudDetection")

    print("AUC:", auc)

2026/05/04 14:02:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 14:02:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGBoost_FraudDetection'.
2026/05/04 14:02:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_FraudDetection, version 1
Created version '1' of model 'XGBoost_FraudDetection'.


AUC: 0.9166059280265445
🏃 View run XGBooxt_Training at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/1/runs/e062886103c3486aad72ab85ce983ed5
🧪 View experiment at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/1
